In [ ]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import json
from sklearn.metrics import roc_auc_score

In [ ]:
CURRENT_FOLDER = Path.cwd()
ROOT_FOLDER = CURRENT_FOLDER.parent
assert (ROOT_FOLDER / 'main.py').exists(), f'not the repo root: {ROOT_FOLDER}'

PARTICIPANT = ROOT_FOLDER / 'recordings' / 'participant001_20260805_092548'


def load_session(session_dir):
    """Load one session's training table and its marker grid layout.

    Returns (df, layout). `layout` is an (n_markers, 2) int array indexed by
    marker_id, so layout[i] is marker i's (row, col) on the sensor.
    """
    df = pd.read_csv(session_dir / 'training_table.csv')

    with (session_dir / 'grid_layout.json').open() as f:
        meta = json.load(f)

    # A fallback_sort layout has no geometric meaning and is written as null.
    if meta['source'] != 'geometry':
        raise ValueError(
            f"{session_dir.name}: grid layout source is {meta['source']!r}, "
            f"not 'geometry' (reason: {meta['reason']})"
        )

    n_rows = meta['grid_shape']['n_rows']
    n_cols = meta['grid_shape']['n_cols']
    n_markers = n_rows * n_cols

    # JSON object keys are always strings, and their order is arbitrary --
    # allocate first, then assign by marker_id.
    layout = np.full((n_markers, 2), -1, dtype=int)
    for marker_id, pos in meta['layout'].items():
        layout[int(marker_id)] = [pos['row'], pos['col']]

    if (layout < 0).any():
        missing = np.flatnonzero(layout[:, 0] < 0).tolist()
        raise ValueError(f'{session_dir.name}: no position for marker ids {missing}')

    return df, layout

In [ ]:
# Converts (3667, 99) marker (dx, dy, etc.) columns to its row-col (3667, 11, 9)

def to_grid(df, prefix, layout):
    n_markers = len(layout)
    n_rows, n_cols = layout.max(axis=0) + 1

    # Generate names in marker_id order so values column i lines up with layout[i].
    cols = [f'{prefix}_{i}' for i in range(n_markers)]
    values = df[cols].to_numpy()

    out = np.full((len(df), n_rows, n_cols), np.nan)
    out[:, layout[:, 0], layout[:, 1]] = values

    assert not np.isnan(out).any(), "out contains NaN values"

    return out

In [ ]:
df, layout = load_session(PARTICIPANT / 'session1_20260805_093706')
grid = to_grid(df, 'dx', layout)

# Verify the scatter against grid_layout.json: each marker's value must land
# at the (row, col) the file assigns it.
assert grid.shape == (len(df), 11, 9), grid.shape
for marker_id, (row, col) in [(0, (0, 0)), (5, (0, 5)), (94, (10, 4)), (98, (10, 8))]:
    assert grid[0, row, col] == df[f'dx_{marker_id}'][0], marker_id

print('to_grid ok:', grid.shape)

In [ ]:
# converting dx, dy, mag, t, and trial to grid

dx = to_grid(df, 'dx', layout)
dy = to_grid(df, 'dy', layout)
mag = to_grid(df, 'mag', layout)
t = df['timestamp_ms'].to_numpy()
trial = df['trial_id'].to_numpy()

print(mag.shape)

In [ ]:
# maximum displacement over magnitude

def max_disp(mag_grid):
    return mag_grid.max(axis=(1,2))

md = max_disp(mag)
np.corrcoef(md, df['force_kg_corr'])[0,1]

In [ ]:
''' sin_theta and cos_theta for direction
    - "out" pre-allocates memory of zeros that has the same dimensions as dy_grid
    - "where" limits division by 0 '''
    
def direction(dx_grid, dy_grid, mag_grid):
    sin_theta = np.divide(dy_grid, mag_grid, out=np.zeros_like(dy_grid), where=mag_grid != 0)
    cos_theta = np.divide(dx_grid, mag_grid, out=np.zeros_like(dx_grid), where=mag_grid != 0)
    return sin_theta, cos_theta

sin_t, cos_t = direction(dx, dy, mag)

print(sin_t.shape, cos_t.shape)

In [ ]:
def strain(dx_grid, dy_grid):
    sx = np.diff(dx_grid, axis=2)
    sx = np.concatenate([sx, sx[:, :, -1:]], axis=2) # axis=2 for column

    sy = np.diff(dy_grid, axis=1)
    sy = np.concatenate([sy, sy[:, -1:, :]], axis=1) # axis=1 for row

    return sx, sy

In [ ]:
def velocity(dx_grid, dy_grid, timestamp_ms, trial_id):
    """Returns (vx, vy) in px/s, same shape as the input grids.

    NaN at the first frame of the session and at every trial boundary --
    there is no valid earlier frame to difference against, and 0 would be
    read by a model as "no movement here".
    """
    dt = np.diff(timestamp_ms) / 1000
    vx = np.diff(dx_grid, axis=0) / dt[:, None, None]
    vy = np.diff(dy_grid, axis=0) / dt[:, None, None]

    # np.diff drops one frame -- prepend NaN so frame i lines up across channels.
    pad = np.full((1,) + dx_grid.shape[1:], np.nan)
    vx = np.concatenate([pad, vx], axis=0)
    vy = np.concatenate([pad, vy], axis=0)

    # A velocity spanning a trial boundary measures the pause, not the muscle.
    boundaries = np.flatnonzero(np.diff(trial_id) != 0) + 1
    vx[boundaries] = np.nan
    vy[boundaries] = np.nan

    return vx, vy


vx, vy = velocity(dx, dy, t, trial)
print(vx.shape, ' NaN frames:', int(np.isnan(vx[:, 0, 0]).sum()))

In [ ]:
def cue_offset(session_dir, df):
    """Seconds from each frame to its trial's grip_start cue. Negative = before it.

    grip_start is the operator's "go" press, logged per trial in events.json.
    It is the real rest/grip boundary: target_pct labels the whole trial, and
    roughly half of every trial is the pre-cue wait, where force sits at
    ~0.7-1.1% MVC no matter what the trial is labelled.
    """
    with (session_dir / 'events.json').open() as f:
        events = json.load(f)['events']

    cue = {e['trial_id']: e['ts'] for e in events if e['type'] == 'grip_start'}

    # NaN for any trial with no cue -- honest, and it fails loudly downstream.
    cue_ts = np.array([cue.get(t, np.nan) for t in df['trial_id'].to_numpy()])

    return (df['timestamp_ms'].to_numpy() - cue_ts) / 1000.0


def build_channels(session_dir):
    """Load one session and derive every candidate channel from it.

    Returns a dict of grids plus the session's own `df` -- later cells need
    `target_pct` and `force_kg_corr`, and they must come from this session.

    Displacement stays RAW: rest offsets are deliberately not subtracted here.
    The "strain vs baseline offset" question measures exactly those offsets, so
    removing them at load time would leave that cell plotting a flat line and
    silently answering nothing.
    """
    df, layout = load_session(session_dir)

    dx_g = to_grid(df, 'dx', layout)
    dy_g = to_grid(df, 'dy', layout)
    mag_g = to_grid(df, 'mag', layout)

    sx, sy = strain(dx_g, dy_g)

    vx_g, vy_g = velocity(dx_g, dy_g, df['timestamp_ms'].to_numpy(), df['trial_id'].to_numpy())

    md = max_disp(mag_g)

    return {
        'df': df,
        'layout': layout,
        'to_cue': cue_offset(session_dir, df),
        'dx': dx_g,
        'dy': dy_g,
        'mag': mag_g,
        'max_disp': md,
        'strain_x': sx,
        'strain_y': sy,
        'vx': vx_g,
        'vy': vy_g,
    }

In [ ]:
# Session 1 = cleanest, session 8 = worst: 1.04 vs 5.18 px resting displacement
# (findings 5, pooled over all rest frames -- post-cue only reads 1.08 / 5.02).
# A channel must survive both.

s1 = build_channels(PARTICIPANT / 'session1_20260805_093706')
s8 = build_channels(PARTICIPANT / 'session8_20260805_105140')

# Shape asserts live here, not inside build_channels -- these numbers are session-specific.
assert s1['dx'].shape == (3667, 11, 9), s1['dx'].shape
assert s8['dx'].shape == (2927, 11, 9), s8['dx'].shape

# Catches the df/grid mismatch a shape check alone cannot see: s8's grids paired with s1's df.
for name, s in (('s1', s1), ('s8', s8)):
    assert len(s['df']) == s['dx'].shape[0], name
    assert len(s['to_cue']) == s['dx'].shape[0], name

# Same r as cell 5 -> the wrapper preserved the loading path we already trust.
print('s1 r(max_disp, force_kg_corr):', np.corrcoef(s1['max_disp'], s1['df']['force_kg_corr'])[0, 1])
print('s8 r(max_disp, force_kg_corr):', np.corrcoef(s8['max_disp'], s8['df']['force_kg_corr'])[0, 1])
print('post-cue frames:', int((s1['to_cue'] >= 0).sum()), int((s8['to_cue'] >= 0).sum()))

In [ ]:
# Velocity noise floor -- does marker speed separate rest from grip at 16 fps?
#
# Both classes are POST-CUE, so they are the same protocol phase and differ only in
# what the participant was asked to do. Labelling by target_pct alone would sweep the
# pre-cue wait into "grip": ~half of every trial, at ~0.7-1.1% MVC regardless of target.
# mag is scored on the identical split, or the number is uninterpretable.

def rest_grip_masks(s, valid=None):
    pct = s['df']['target_pct'].to_numpy()
    post = s['to_cue'] >= 0                               # after the operator's "go" press
    ok = ~np.isnan(s['to_cue'])                           # trials with no grip_start event
    if valid is not None:
        ok = ok & valid                                   # channel-specific NaNs, e.g. velocity
    return (pct == 0) & post & ok, (pct == 80) & post & ok


def rest_grip_auc(score_grid, is_rest, is_grip):
    rest = score_grid[is_rest].ravel()                    # 1-D mask picks whole frames -> 99 each
    grip = score_grid[is_grip].ravel()

    y = np.concatenate([np.zeros(rest.size), np.ones(grip.size)])   # 0 = rest, 1 = grip
    scores = np.concatenate([rest, grip])                 # higher must mean "more grip-like"

    return roc_auc_score(y, scores), rest, grip


speeds = {}                                               # kept for the histogram below

for name, s in (('s1', s1), ('s8', s8)):
    speed = np.sqrt(s['vx'] ** 2 + s['vy'] ** 2)          # unsigned: vx flips sign on release

    valid = ~np.isnan(speed).any(axis=(1, 2))             # frame 0 + every trial boundary
    is_rest, is_grip = rest_grip_masks(s, valid)

    assert not np.isnan(speed[is_rest | is_grip]).any()   # roc_auc_score raises on NaN, not skips

    auc_v, v_rest, v_grip = rest_grip_auc(speed, is_rest, is_grip)      # channel under test
    auc_m, _, _ = rest_grip_auc(s['mag'], is_rest, is_grip)             # reference, same split

    speeds[name] = (v_rest, v_grip)

    print(f'{name}: {is_rest.sum()} rest / {is_grip.sum()} grip frames (post-cue)')
    print(f'    AUC   speed {auc_v:.4f}    mag {auc_m:.4f}')
    print(f'    speed px/s  rest  median {np.median(v_rest):6.2f}  p90 {np.percentile(v_rest, 90):7.2f}')
    print(f'                grip  median {np.median(v_grip):6.2f}  p90 {np.percentile(v_grip, 90):7.2f}')

# Medians equal + p90 far apart = non-monotonic (ramp bursts, quiet hold), not pure noise.
fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
bins = np.logspace(-2, 3, 60)                             # log bins: speeds are heavy-tailed

for ax, (name, (v_rest, v_grip)) in zip(axes, speeds.items()):
    floor = 1e-2                                          # exact 0 exists (3-dec storage); log drops it
    ax.hist(np.clip(v_rest, floor, None), bins=bins, alpha=0.6, density=True, label='rest (0%)')
    ax.hist(np.clip(v_grip, floor, None), bins=bins, alpha=0.6, density=True, label='grip (80%)')
    ax.set_xscale('log')
    ax.set_title(name)
    ax.set_xlabel('marker speed (px/s)')

axes[0].set_ylabel('density')
axes[0].legend()
plt.tight_layout()

In [ ]:
# Strain separation -- better, worse, or just different from raw dx/dy?
# Same post-cue split as the velocity cell, so every number here is comparable to it.
# abs() throughout: AUC needs "higher = more grip-like", and strain/dx/dy are signed.

for name, s in (('s1', s1), ('s8', s8)):
    is_rest, is_grip = rest_grip_masks(s)

    strain_mag = np.sqrt(s['strain_x'] ** 2 + s['strain_y'] ** 2)
    chans = {
        'mag': s['mag'],                                  # reference channel
        '|dx|': np.abs(s['dx']),
        '|dy|': np.abs(s['dy']),
        '|strain_x|': np.abs(s['strain_x']),
        '|strain_y|': np.abs(s['strain_y']),
        'strain_mag': strain_mag,
    }

    aucs = {c: rest_grip_auc(g, is_rest, is_grip)[0] for c, g in chans.items()}

    # AUC cannot see redundancy -- strain is computed FROM dx/dy. Correlation can.
    pooled = is_rest | is_grip
    r = np.corrcoef(strain_mag[pooled].ravel(), s['mag'][pooled].ravel())[0, 1]

    print(f'{name}: {is_rest.sum()} rest / {is_grip.sum()} grip frames')
    for c, a in sorted(aucs.items(), key=lambda kv: -kv[1]):
        print(f'    {c:<12} AUC {a:.4f}')
    print(f'    corr(strain_mag, mag) = {r:+.4f}   <- high means strain restates displacement')

In [ ]:
# Strain vs the baseline-offset problem.
# Session 8 rests ~5x higher than session 1 (findings 5). A rigid whole-grid shift adds
# the SAME constant to every dx, and strain is a neighbour difference -- so a purely rigid
# offset cancels exactly. Expected: strain shrinks the s8/s1 gap but does not close it,
# because findings 5 puts the rigid part at only 12-41%. Record the actual number.

rest_stats = {}

for name, s in (('s1', s1), ('s8', s8)):
    is_rest, _ = rest_grip_masks(s)                       # post-cue 0% frames only

    disp = np.median(s['mag'][is_rest])
    strn = np.median(np.sqrt(s['strain_x'] ** 2 + s['strain_y'] ** 2)[is_rest])

    # Rigid part = the whole-grid mean shift per frame; the rest is local deformation.
    rig = np.hypot(s['dx'][is_rest].mean(axis=(1, 2)), s['dy'][is_rest].mean(axis=(1, 2)))
    rig_frac = np.median(rig) / disp

    rest_stats[name] = (disp, strn)
    print(f'{name}: rest frames {is_rest.sum():4d}   median disp {disp:5.2f} px   '
          f'median strain {strn:5.2f} px   rigid share {rig_frac:5.1%}')

# The gap between sessions, measured in each space. Ratio, so units cancel.
gap_disp = rest_stats['s8'][0] / rest_stats['s1'][0]
gap_strain = rest_stats['s8'][1] / rest_stats['s1'][1]

print(f'\ns8/s1 resting gap   displacement {gap_disp:.2f}x   strain {gap_strain:.2f}x')
print(f'strain closes {1 - (gap_strain - 1) / (gap_disp - 1):.1%} of the gap '
      f'({gap_disp:.2f}x -> {gap_strain:.2f}x)')

In [ ]:
# Strain edge seam.
# The one-sided fallback COPIES the last real difference, so the last column of strain_x
# is identical to the second-to-last by construction -- there can be no "jump" to plot.
# The real question is how much that forced equality distorts the map: neighbouring
# columns normally differ, and at the seam we impose a difference of exactly zero.

for name, s in (('s1', s1), ('s8', s8)):
    sx, sy = s['strain_x'], s['strain_y']

    assert np.array_equal(sx[:, :, -1], sx[:, :, -2]), name    # duplication is exact
    assert np.array_equal(sy[:, -1, :], sy[:, -2, :]), name

    # How different are genuinely adjacent columns/rows? The seam is forced to 0.
    col_steps = [np.median(np.abs(sx[:, :, c + 1] - sx[:, :, c])) for c in range(sx.shape[2] - 2)]
    row_steps = [np.median(np.abs(sy[:, r + 1, :] - sy[:, r, :])) for r in range(sy.shape[1] - 2)]

    spread = np.median(np.abs(sx))                        # typical strain size, for scale
    print(f'{name}: typical |strain_x| {spread:.3f} px')
    print(f'    natural column-to-column step  median {np.median(col_steps):.3f} px  '
          f'(range {min(col_steps):.3f}-{max(col_steps):.3f})')
    print(f'    natural row-to-row step        median {np.median(row_steps):.3f} px  '
          f'(range {min(row_steps):.3f}-{max(row_steps):.3f})')
    print(f'    seam step is 0.000 px by construction -> '
          f'{11 / 99:.0%} of markers carry a copied strain_x, {9 / 99:.0%} a copied strain_y')

# Visual: per-column spread of strain_x. The last two boxes must be identical.
fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)

for ax, (name, s) in zip(axes, (('s1', s1), ('s8', s8))):
    is_rest, is_grip = rest_grip_masks(s)
    grip_sx = s['strain_x'][is_grip]                      # grip frames: strain is largest here
    ax.boxplot([grip_sx[:, :, c].ravel() for c in range(grip_sx.shape[2])], showfliers=False)
    ax.axvline(grip_sx.shape[2] - 0.5, color='r', ls='--', lw=1, label='copied column')
    ax.set_title(name)
    ax.set_xlabel('grid column')

axes[0].set_ylabel('strain_x (px)')
axes[0].legend()
plt.tight_layout()

## Verdict — which channels earn a column

**Sessions:** 1 (cleanest, 1.08 px rest) and 8 (worst, 5.02 px rest). A channel had to survive both.

**Split:** rest = `target_pct == 0` post-cue, grip = `target_pct == 80` post-cue. Both classes sit in
the same protocol phase, so they differ only in what the participant was asked to do. Labelling by
`target_pct` alone would have swept the pre-cue wait into "grip" — about half of every trial, at
~0.7–1.1% MVC regardless of target.

| Channel | Verdict | Evidence |
|---|---|---|
| `max_disp` | **KEEP** | r = 0.8171 (s1) / 0.6441 (s8) vs `force_kg_corr`, matching `sync_report.json` exactly. One column. |
| `sin_theta`, `cos_theta` | **CUT** | Decided on argument, not measurement — zero information added over `dx`/`dy`, and numerically corrupt at rest where `mag` is small (3-decimal storage). |
| `vx`, `vy` | **NOT OBVIOUSLY DEAD** | Speed AUC 0.8180 (s1) / 0.7655 (s8), beating the `mag` reference (0.7216 / 0.6366) on both. |
| `strain_x`, `strain_y` | **NOT OBVIOUSLY DEAD** | `strain_mag` AUC 0.7466 (s1) / 0.6083 (s8). Beats raw `|dx|`/`|dy|` on both sessions. |

### Velocity — the channel that was expected to die

The plan called this the weakest of the six and told us to try to kill it. It survived: speed
outscores `mag` on both sessions, and the rest/grip medians are 1.23 → 4.60 px/s (s1) and
1.26 → 4.15 px/s (s8), with p90 separating 3–5× further.

Two reasons that is **not** the same as "useful":

- **AUC cannot detect redundancy.** Speed is computed from `dx`/`dy`, exactly as `mag` is. Beating
  `mag` does not show it carries new information.
- **`mag` is a handicapped reference**, badly so on session 8, where rest sits at 5.02 px. Velocity
  differences cancel a constant baseline offset; `mag` does not. Some of the margin is `mag` being
  penalised rather than speed being strong.

### Strain — mixed on AUC, but genuinely different

`strain_mag` beats `mag` on session 1 (0.7466 vs 0.7216) and loses on session 8 (0.6083 vs 0.6366).
Componentwise it is weaker still — `|strain_x|` (0.5760) slightly loses to `|dx|` (0.5767) on session 8.

The number that matters more is the correlation the AUC could not see:

**corr(`strain_mag`, `mag`) = +0.579 (s1), +0.331 (s8).**

Moderate, not high. Strain is **not** a restatement of displacement — most of its variation is its own.
That is the argument for keeping it, and it is the one the scope revision predicted: Deep Sets sees an
unordered bag of markers with no adjacency, and strain is the only route by which spatial structure
reaches it.

### Strain and the baseline-offset problem

|  | session 1 | session 8 | gap |
|---|---|---|---|
| median resting displacement | 1.08 px | 5.02 px | **4.65×** |
| median resting strain | 0.54 px | 1.61 px | **2.99×** |
| rigid share of resting motion | 31.3% | 20.9% | — |

**Strain closes 45.5% of the between-session gap, and leaves a 2.99× gap standing.** That is the
honest version of a claim that is easy to oversell: strain helps with the baseline-offset problem, but
it is **not** a substitute for per-session normalisation.

The measured rigid share (20.9–31.3%) sits inside the 12–41% range in `data-quality-findings.md` §5,
which is independent corroboration. Note that strain removes rather more of the gap than the rigid
share alone predicts — it suppresses smooth, low-spatial-frequency drift too, not only perfectly
rigid shift.

### Strain edge seam — no discontinuity, but an information hole

There can be no visible jump: the one-sided fallback **copies** the last real difference, so the last
column of `strain_x` is bit-for-bit identical to the second-to-last (asserted in the cell above). The
real cost is what that copy hides:

| | session 1 | session 8 |
|---|---|---|
| typical `\|strain_x\|` | 0.560 px | 1.172 px |
| natural column-to-column step | 0.444 px | 0.590 px |
| step imposed at the seam | **0.000 px** | **0.000 px** |

Genuinely adjacent columns differ by ~79% (s1) of the typical strain magnitude. At the seam we force
that to zero. So **11% of markers carry a copied `strain_x` and 9% a copied `strain_y`** — those cells
are not estimates, they are duplicates.

Not a reason to change the edge rule, which plan §1 settled: every marker keeps a value, and the
alternative (a shorter grid) loses a column outright. But it is a documented artifact, and any
Grad-CAM heatmap that lights up the last column or row should be read with it in mind.

### What this does not settle

- **Redundancy.** AUC compares channels one at a time; it cannot say whether strain adds anything
  *on top of* `dx`/`dy`. That is the ablation planned for notebook 03, on the inner fold only.
- **Per-model value.** Strain is provably redundant for Ridge (a linear function of `dx`), learnable
  by a 2D CNN (it is a `[-1, +1]` kernel), and underivable by Deep Sets. The keep/cut answer is
  per-model and belongs in notebooks 03/06.
- **Sample independence.** ~24,000–29,000 per-marker values come from a few hundred frames, and the
  99 markers in a frame move together. These AUCs are descriptive — no p-values, no confidence
  intervals.
- **n = 1 participant.** Everything here is participant001. Session 1 and session 8 disagree often
  enough that a second participant could move any of these numbers.

### Carried into Phase 2 (`core/features.py`)

`max_disp`, `strain_x`, `strain_y`, `vx`, `vy`. `sin_theta` / `cos_theta` are not built.